# MARV × Titans — the complete walkthrough (Colab)

**The question:** MARV finds where a fact lives in a frozen model's weights, edits it, and measures the damage. Titans' neural memory is different from a frozen model — it's an MLP that keeps rewriting itself while the model reads. Can the same find-it/edit-it/measure-it discipline work on a memory that's constantly changing?

**This notebook runs every experiment that answered that question, in the order they actually happened.** Sections A–D are fast (seconds, no GPU needed — pure CPU is fine). Section E trains a real small language model and benefits from a GPU (`Runtime → Change runtime type → T4 GPU`), taking ~10-12 minutes there instead of longer on CPU.

| Section | What it shows | Needs GPU? |
|---|---|---|
| A | An untrained memory doesn't forget; a memory trained on a toy task forgets fast | No |
| B | No single unit stores one fact — proven by deleting units, not guessed | No |
| C | Give units their own forget rate, and real localization appears | No |
| D | Edit a fact directly — exact edits, with real collateral damage | No |
| E | Train on REAL text: the forgetting curve reverses | Recommended |

Full write-up: `experiments/README.md` on the `marv-titan` branch of [github.com/thebnbrkr/marv](https://github.com/thebnbrkr/marv).

In [ ]:
!pip install -q titans-pytorch
!git clone -q -b marv-titan https://github.com/thebnbrkr/marv.git /content/marv
import sys; sys.path.insert(0, '/content/marv/experiments')

import torch, numpy as np, matplotlib.pyplot as plt
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

---
## A. The forgetting curve, on random vectors

A Titans `NeuralMemory` (`dim=64 → hidden=256 → dim=64`) reads a document of random 64-dim vectors. Snapshot its weights early and late, diff them per hidden unit — the same move `marv.diff` makes on two training checkpoints, just applied to a memory that changes *within* one forward pass instead of between two separate checkpoints.

In [ ]:
from titans_pytorch import NeuralMemory
from titans_memdiff import DIM, HIDDEN, CHUNK, DOC_LEN, train_recall, snapshots, _cos

torch.manual_seed(0); np.random.seed(0)
mem_raw = NeuralMemory(dim=DIM, chunk_size=CHUNK).to(device)
mem_trained = NeuralMemory(dim=DIM, chunk_size=CHUNK).to(device)
print('training a memory on autoassociative recall (a toy task -- query with x, should return x)...')
train_recall(mem_trained, steps=400, device=device)

In [ ]:
def run_diff(mem, seq):
    U0, U1, vals = snapshots(mem, seq)
    g_in, g_out = U0[1], U0[-1]
    nr = (np.linalg.norm(g_out, axis=0) + 1e-9) / (np.linalg.norm(g_in, axis=0) + 1e-9)
    return nr

seq = torch.randn(1, DOC_LEN, DIM, device=device)
nr_raw = run_diff(mem_raw, seq)
nr_trained = run_diff(mem_trained, seq)
print(f'UNTRAINED memory  norm_ratio (end/early write): mean {nr_raw.mean():.2f}  -- writes ACCUMULATE')
print(f'TRAINED memory    norm_ratio (end/early write): mean {nr_trained.mean():.2f}  -- writes DECAY hard')

**What to look for:** the untrained memory's ratio should be well above 1 (writes pile up); the trained one's should be a small fraction (writes decay fast). This one toy task teaches the memory an aggressive forget habit — Section E shows that's specific to THIS task, not memories in general.

---
## B. No single unit stores one fact — proven, not guessed

Store distinct tracked key/value pairs, then delete one hidden unit at a time and watch whether any single pair's recall breaks. Fixes an earlier blocker (a hand-rolled retrieval call only matched the real model at cos ≈ 0.6) by calling the library's own `retrieve_memories()` -- verified to reproduce the true output at cos ≈ 1.0.

In [ ]:
from titans_ablation import (
    DIM as DIM_B, HIDDEN as HIDDEN_B, BREAK_THRESHOLD,
    store_tracked_pairs, run_ablation_sweep, check_replay_fidelity,
)

torch.manual_seed(0)
mem = NeuralMemory(dim=DIM_B, chunk_size=1).to(device)
train_recall(mem, steps=400, device=device)

fidelity = check_replay_fidelity(mem, torch.randn(1, 20, DIM_B, device=device))
print(f'replay-fidelity check (want ~1.0): {fidelity:.4f}')

pairs, weights = store_tracked_pairs(mem, 12, DIM_B, device, seed=0)
baseline, drop = run_ablation_sweep(mem, weights, pairs)
hit = np.abs(drop) > BREAK_THRESHOLD
print(f'\nlargest single-unit effect on any pair: {np.abs(drop).max():.3f}  (threshold {BREAK_THRESHOLD})')
print(f'units breaking 0 / 1 / >1 pairs cleanly: {(hit.sum(1)==0).sum()} / {(hit.sum(1)==1).sum()} / {(hit.sum(1)>1).sum()}')

**What to look for:** the largest effect should be small (well under the break threshold), and almost every unit breaks zero pairs cleanly. No single memory slot is "the" home for any one fact in this architecture.

---
## C. Why — and the fix: give units their own dial

Reading the memory's real update rule shows the forget gate is ONE shared number applied to all 256 units identically -- confirmed directly in the paper (arXiv:2501.00663, eq. 13 in the main text; Appendix C's `diag(1-alpha_t)` shows the general form *would* allow per-unit decay, but the released code doesn't use it). This tests that directly: assign each unit its OWN fixed decay rate instead of learning one end-to-end (an earlier attempt at learning it didn't converge), and see if real localization appears.

In [ ]:
from titans_per_unit import (
    HIDDEN as HIDDEN_C, DIM as DIM_C, store_tracked_pairs as store_pairs_c,
    run_ablation_sweep as ablation_sweep_c, report,
)

n_pairs = 12
seed = 0
g = torch.Generator().manual_seed(seed)
pairs_c = torch.randn(n_pairs, DIM_C, generator=g)

uniform_decay = torch.full((HIDDEN_C,), 0.3)
spread_decay = torch.linspace(0.02, 0.98, HIDDEN_C)

for label, decay in [('UNIFORM decay (mirrors the real architecture)', uniform_decay),
                      ('SPREAD decay (mirrors what diag(1-alpha_t) allows)', spread_decay)]:
    w0, w1 = store_pairs_c(pairs_c, decay, seed)
    baseline_c, drop_c = ablation_sweep_c(w0, w1, pairs_c)
    report(label, decay, baseline_c, drop_c)
    print()

**What to look for:** under uniform decay, no unit should ever cleanly break exactly one pair. Under spread decay, some units should — a slow-forgetting unit becomes the de facto home for one specific fact. Confirmed across 8 independent seeds in the full writeup: 0/8 under uniform, 4/8 under spread, and even the non-crossing seeds ran 5-7× stronger than uniform's ceiling every time.

---
## D. Edit a fact directly — MARV's actual core move

Not just deletion: solve for a new output row on the unit that owns a fact, so it recalls a completely different target instead — using only that one unit's weights.

In [ ]:
from titans_per_unit import run_editing_demo

print('Editing pair 0 (owned by unit 4) to recall pair 5\'s value instead:')
run_editing_demo(pairs_c, spread_decay, seed=seed, unit=4, edit_pair_idx=0, target=pairs_c[5])

**What to look for:** the edited pair's recall of its new target should be ~1.000 (the edit is always mathematically exact). The real finding is the `<-- collateral` rows — several *other* facts move too, even though only one unit was touched. In the full writeup, picking a unit that looks even MORE isolated by the ablation test (nearly 2× more "specific") made the collateral *worse*, not better — ablation-specificity and editing-safety turn out to be different properties of the same unit.

---
## E. Train on REAL text — the forgetting curve reverses

Everything above used random vectors, on purpose, to isolate the memory mechanism from language modeling. This section trains a small, real, byte-level language model with the SAME kind of Titans memory wired in (`titans_pytorch.MemoryAsContextTransformer`), on real Wikipedia text (enwik8), then re-runs the same early-vs-end diff on a real held-out passage.

Scaled way down from the library's own recipe on purpose (dim 64 to match the memory's own width, one memory layer, no flex-attention) — this will not be a good language model, it's a correctness + qualitative-structure check. **Recommend switching to a T4 GPU for this section** (`Runtime → Change runtime type`).

In [ ]:
!git clone -q --depth 1 https://github.com/lucidrains/titans-pytorch.git /content/titans-pytorch-src
import os
DATA_PATH = '/content/titans-pytorch-src/data/enwik8.gz'
assert os.path.exists(DATA_PATH), 'enwik8.gz not found -- see http://prize.hutter1.net/ for a fallback source'

from titans_real_text import build_model, load_enwik8, sample_batch, train, diff_memory_on_passage

data_train, data_val = load_enwik8(DATA_PATH)
model = build_model().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'model: {n_params/1e6:.2f}M params, device={device}')

In [ ]:
STEPS = 3000       # ~10-12 min on a T4; raise it if you want a (still small) stronger model
SEQ_LEN = 256
BATCH_SIZE = 16

train(model, data_train, data_val, steps=STEPS, seq_len=SEQ_LEN, batch_size=BATCH_SIZE, lr=2e-4, device=device)

In [ ]:
passage = sample_batch(data_val, 512, 1)[0]
print('passage:', repr(bytes(passage[:150].tolist()).decode('utf-8', errors='replace')))
print()
diff_memory_on_passage(model, passage, device)
print(f'\ncompare to Section A\'s TRAINED-on-toy-task norm_ratio: {nr_trained.mean():.2f}')

**What to look for:** `norm_ratio` here should land well ABOVE 1 (writes accumulating) — the opposite of Section A's toy-task-trained memory. Confirmed across 5 independent runs (local CPU and Colab T4, at both 1000-1200 and 3000 steps): the effect gets STRONGER with more training, ruling out "just an undertrained artifact." A memory trained on real language modeling does not develop the aggressive forgetting seen on the toy recall task — plausibly because predicting the next byte of real text rewards retaining context, while the toy task had nothing to gain from keeping anything beyond the immediate query.

---
## What this all adds up to

1. **A memory trained on a toy task forgets fast; the same architecture trained on real text accumulates instead** — the toy-task result was real, but specific to that task, not a general property of trained Titans memories.
2. **No single unit stores one fact in the real architecture** — proven by actually deleting units and watching nothing break cleanly, not inferred from a proxy.
3. **That's a consequence of the forget gate being one shared dial for all units**, confirmed directly against the paper's own equations — not an emergent mystery.
4. **Give units their own dial, and real localization appears** — the missing capability, not an inherent limit of test-time memory.
5. **You genuinely can edit a fact in a live memory the way MARV edits a frozen one** — exact edits are possible, and they carry the same collateral-damage cost MARV already documents for frozen models.

Open questions and the full run-by-run data: `experiments/README.md` on the `marv-titan` branch.